[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter4/caching.ipynb)

# Chapter 4: Caching in a LangChain RAG Pipeline

This notebook demonstrates two approaches to retrieval caching in a LangChain RAG pipeline:

1. **Exact-Match Caching**: Uses hash-based lookup - only identical queries hit the cache
2. **Semantic Caching**: Uses embedding similarity - semantically similar queries can hit the cache

We'll see how semantic caching provides better cache utilization when users ask the same question in different ways.

**What you'll learn:**
- Implement exact-match caching with Redis for a LangChain retriever
- Build a semantic cache using embedding similarity for fuzzy query matching
- Compare cache hit behavior between exact-match and semantic approaches

**Prerequisites:** Docker must be installed and running (Redis will be started automatically). `pip install langchain langchain-openai faiss-cpu redis` and set `OPENAI_API_KEY`.

In [1]:
import os
import redis
import hashlib
import json
import numpy as np
from typing import List, Optional, Tuple

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import PrivateAttr

In [2]:
# Start Redis container (using Docker)
import subprocess
import time

# Remove existing container if present, then start fresh
subprocess.run(["docker", "rm", "-f", "redis-cache"], capture_output=True)
subprocess.run(["docker", "run", "-d", "--name", "redis-cache", "-p", "6379:6379", "redis"], check=True)

# Wait for Redis to be ready
time.sleep(2)
print("Redis container started.")

Unable to find image 'redis:latest' locally
latest: Pulling from library/redis
33bdc9671af8: Pulling fs layer
78e7451da07b: Pulling fs layer
7e32205e89f6: Pulling fs layer
45291b6843ab: Pulling fs layer
02f8f8d6b1ad: Pulling fs layer
4f4fb700ef54: Pulling fs layer
fee7b1b2fd22: Pulling fs layer
02f8f8d6b1ad: Waiting
4f4fb700ef54: Waiting
fee7b1b2fd22: Waiting
45291b6843ab: Waiting
78e7451da07b: Verifying Checksum
78e7451da07b: Download complete
7e32205e89f6: Verifying Checksum
7e32205e89f6: Download complete
02f8f8d6b1ad: Verifying Checksum
02f8f8d6b1ad: Download complete
33bdc9671af8: Verifying Checksum
33bdc9671af8: Download complete
4f4fb700ef54: Verifying Checksum
4f4fb700ef54: Download complete
45291b6843ab: Verifying Checksum
45291b6843ab: Download complete
fee7b1b2fd22: Verifying Checksum
fee7b1b2fd22: Download complete
33bdc9671af8: Pull complete
78e7451da07b: Pull complete
7e32205e89f6: Pull complete
45291b6843ab: Pull complete
02f8f8d6b1ad: Pull complete
4f4fb700ef54: Pull co

8d4704ba69c7482cc627711b89196fd5276c8fd1ce8107cb9f27cd03adcaa38b
Redis container started.


In [3]:
# Initialize Redis cache
cache = redis.Redis(host='localhost', port=6379)

## Part 1: Exact-Match Caching

This approach hashes the query string and uses it as a cache key. Only character-for-character identical queries will hit the cache.

In [4]:
class CachedRetriever(BaseRetriever):
    """
    A LangChain retriever that wraps another retriever with Redis caching.
    """
    _base_retriever: any = PrivateAttr()
    _cache_ttl: int = PrivateAttr(default=3600)
    
    def __init__(self, base_retriever, cache_ttl: int = 3600, **kwargs):
        super().__init__(**kwargs)
        self._base_retriever = base_retriever
        self._cache_ttl = cache_ttl
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        # Create a deterministic hash of the query
        query_hash = hashlib.sha256(query.encode('utf-8')).hexdigest()
        
        # Check fast cache (Redis)
        cached_result = cache.get(query_hash)
        if cached_result:
            print("Cache Hit! Skipping vector search.")
            docs_data = json.loads(cached_result)
            return [Document(page_content=d['content'], metadata=d['metadata']) for d in docs_data]

        # Cache Miss: Perform the expensive vector search
        print("Cache Miss. Performing vector search...")
        results = self._base_retriever.invoke(query)
        
        # Serialize and store in cache with TTL
        docs_data = [{'content': doc.page_content, 'metadata': doc.metadata} for doc in results]
        cache.setex(query_hash, self._cache_ttl, json.dumps(docs_data))
        
        return results

## Part 2: Semantic Caching

This approach uses embedding similarity to find cached results. Semantically similar queries (e.g., "What is RAG?" and "Tell me about RAG") can hit the cache even if the wording differs.

Key components:
- Store query embeddings alongside cached results
- On new queries, compute similarity to cached queries  
- Return cached results if similarity exceeds threshold

In [5]:
class SemanticCachedRetriever(BaseRetriever):
    """
    A retriever that uses embedding similarity for semantic caching.
    Similar queries can hit the cache even if worded differently.
    """
    _base_retriever: any = PrivateAttr()
    _embeddings: any = PrivateAttr()
    _similarity_threshold: float = PrivateAttr(default=0.85)
    _cache_embeddings: List[np.ndarray] = PrivateAttr(default_factory=list)
    _cache_results: List[List[Document]] = PrivateAttr(default_factory=list)
    _cache_queries: List[str] = PrivateAttr(default_factory=list)
    
    def __init__(
        self, 
        base_retriever, 
        embeddings,
        similarity_threshold: float = 0.85,
        **kwargs
    ):
        super().__init__(**kwargs)
        self._base_retriever = base_retriever
        self._embeddings = embeddings
        self._similarity_threshold = similarity_threshold
        self._cache_embeddings = []
        self._cache_results = []
        self._cache_queries = []
    
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine similarity between two vectors."""
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    def _find_similar_cached(
        self, query_embedding: np.ndarray
    ) -> Optional[Tuple[List[Document], str, float]]:
        """Find cached results for a semantically similar query."""
        best_similarity = 0.0
        best_match = None
        best_query = None
        
        for i, cached_emb in enumerate(self._cache_embeddings):
            similarity = self._cosine_similarity(query_embedding, cached_emb)
            if similarity > best_similarity:
                best_similarity = similarity
                best_match = self._cache_results[i]
                best_query = self._cache_queries[i]
        
        if best_similarity >= self._similarity_threshold:
            return best_match, best_query, best_similarity
        return None
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        # Compute embedding for the query
        query_embedding = np.array(self._embeddings.embed_query(query))
        
        # Check for semantically similar cached query
        cache_hit = self._find_similar_cached(query_embedding)
        if cache_hit:
            docs, original_query, similarity = cache_hit
            print(f"Semantic Cache Hit! (similarity: {similarity:.3f})")
            print(f"  Matched query: \"{original_query}\"")
            return docs
        
        # Cache miss - perform retrieval
        print("Semantic Cache Miss. Performing vector search...")
        results = self._base_retriever.invoke(query)
        
        # Store in cache
        self._cache_embeddings.append(query_embedding)
        self._cache_results.append(results)
        self._cache_queries.append(query)
        
        return results
    
    def clear_cache(self):
        """Clear the semantic cache."""
        self._cache_embeddings = []
        self._cache_results = []
        self._cache_queries = []

> **Note:** The `similarity_threshold` controls how close a new query must be to a cached query to return the cached answer. A value of 0.85 is moderately strict — it allows minor rephrasings (e.g., "What is RAG?" vs. "Explain RAG") to hit the cache while avoiding false matches. Lower the threshold to cache more aggressively (risking stale answers); raise it for higher precision.

In [6]:
# Create sample documents for the demo
sample_text = """
Artificial Intelligence (AI) is transforming industries worldwide.
Machine learning models can analyze vast amounts of data to find patterns.
Deep learning uses neural networks with multiple layers to learn representations.
Natural Language Processing (NLP) enables computers to understand human language.
Retrieval-Augmented Generation (RAG) combines retrieval systems with language models.
RAG helps reduce hallucinations by grounding responses in retrieved documents.
Vector databases store embeddings for efficient similarity search.
Caching can significantly improve RAG system performance by avoiding redundant searches.
"""

# Save sample text to a file
with open("sample_docs.txt", "w") as f:
    f.write(sample_text)

print("Sample documents created.")

Sample documents created.


In [7]:
# Initialize LLM and embeddings
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

# Load and split documents
loader = TextLoader("sample_docs.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
texts = text_splitter.split_documents(docs)

print(f"Created {len(texts)} text chunks")

Created 4 text chunks


In [8]:
# Create vector store and base retriever
vectorstore = FAISS.from_documents(texts, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Wrap with caching
cached_retriever = CachedRetriever(base_retriever, cache_ttl=3600)

print("Vector store and cached retriever initialized.")

Vector store and cached retriever initialized.


In [9]:
# Create the RAG chain with cached retriever using LCEL
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": cached_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created with cached retriever.")

RAG chain created with cached retriever.


In [10]:
# Clear any existing cache for demo purposes
query = "What is RAG and how does it help?"
query_hash = hashlib.sha256(query.encode('utf-8')).hexdigest()
cache.delete(query_hash)

print("Cache cleared for demo query.")

Cache cleared for demo query.


In [11]:
# First query - Cache Miss (performs vector search)
print("=== First Query (Cache Miss) ===")
response = rag_chain.invoke(query)
print(f"\nAnswer: {response}")

=== First Query (Cache Miss) ===
Cache Miss. Performing vector search...

Answer: Retrieval-Augmented Generation (RAG) is a system that combines retrieval mechanisms with language models. It helps reduce hallucinations in generated responses by grounding them in retrieved documents, ensuring that the information provided is more accurate and reliable.


In [12]:
# Second query - Cache Hit (skips vector search)
print("=== Second Query (Cache Hit) ===")
response = rag_chain.invoke(query)
print(f"\nAnswer: {response}")

=== Second Query (Cache Hit) ===
Cache Hit! Skipping vector search.

Answer: Retrieval-Augmented Generation (RAG) is a system that combines retrieval mechanisms with language models. It helps reduce hallucinations in generated responses by grounding them in retrieved documents, ensuring that the information provided is more accurate and reliable.


## Part 3: Comparison - The Power of Semantic Caching

Now let's see the key difference. We'll ask a semantically similar question with different wording.

With **exact-match caching**, this will be a cache miss (different hash).
With **semantic caching**, this should be a cache hit (similar embedding).

In [13]:
# Define our test queries
original_query = "What is RAG and how does it help?"
similar_query = "Tell me about RAG and its benefits"

print(f"Original query: \"{original_query}\"")
print(f"Similar query:  \"{similar_query}\"")
print(f"\nThese ask the same thing but with different wording.")

Original query: "What is RAG and how does it help?"
Similar query:  "Tell me about RAG and its benefits"

These ask the same thing but with different wording.


In [14]:
# Test exact-match cache with the similar query
print("=== Exact-Match Cache Test ===")
print(f"Query: \"{similar_query}\"\n")

# This will miss because the hash is different
_ = cached_retriever.invoke(similar_query)

print("\n❌ Exact-match cache missed - had to perform vector search again!")

=== Exact-Match Cache Test ===
Query: "Tell me about RAG and its benefits"

Cache Miss. Performing vector search...

❌ Exact-match cache missed - had to perform vector search again!


In [15]:
# Create semantic cached retriever
semantic_retriever = SemanticCachedRetriever(
    base_retriever, 
    embeddings,
    similarity_threshold=0.85
)

# First, prime the cache with the original query
print("=== Priming Semantic Cache ===")
print(f"Query: \"{original_query}\"\n")
_ = semantic_retriever.invoke(original_query)

=== Priming Semantic Cache ===
Query: "What is RAG and how does it help?"

Semantic Cache Miss. Performing vector search...


In [16]:
# Now test with the semantically similar query
print("=== Semantic Cache Test ===")
print(f"Query: \"{similar_query}\"\n")
_ = semantic_retriever.invoke(similar_query)

print("\n✓ Semantic cache hit - no vector search needed!")

=== Semantic Cache Test ===
Query: "Tell me about RAG and its benefits"

Semantic Cache Hit! (similarity: 0.947)
  Matched query: "What is RAG and how does it help?"

✓ Semantic cache hit - no vector search needed!


In [17]:
# Try another semantically similar query
third_query = "Explain what RAG is and why it's useful"
print("=== Another Semantic Cache Test ===")
print(f"Query: \"{third_query}\"\n")
_ = semantic_retriever.invoke(third_query)

print("\n✓ Another hit! The cache recognizes this is asking the same thing.")

=== Another Semantic Cache Test ===
Query: "Explain what RAG is and why it's useful"

Semantic Cache Hit! (similarity: 0.954)
  Matched query: "What is RAG and how does it help?"

✓ Another hit! The cache recognizes this is asking the same thing.


In [18]:
# Test with a genuinely different query - should miss
different_query = "What are vector databases?"
print("=== Different Topic Query ===")
print(f"Query: \"{different_query}\"\n")
_ = semantic_retriever.invoke(different_query)

print("\nThis query is about a different topic, so it correctly misses the cache.")

=== Different Topic Query ===
Query: "What are vector databases?"

Semantic Cache Miss. Performing vector search...

This query is about a different topic, so it correctly misses the cache.


In [19]:
# Build a RAG chain with semantic caching
semantic_rag_chain = (
    {"context": semantic_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Clear the semantic cache for a fresh demo
semantic_retriever.clear_cache()

print("=== Full RAG Chain with Semantic Caching ===\n")

# First query
print(f"Query 1: \"{original_query}\"")
response = semantic_rag_chain.invoke(original_query)
print(f"Answer: {response}\n")

# Semantically similar query - should hit cache
print(f"Query 2: \"{similar_query}\"")
response = semantic_rag_chain.invoke(similar_query)
print(f"Answer: {response}")

=== Full RAG Chain with Semantic Caching ===

Query 1: "What is RAG and how does it help?"
Semantic Cache Miss. Performing vector search...
Answer: Retrieval-Augmented Generation (RAG) is a system that combines retrieval mechanisms with language models. It helps reduce hallucinations in generated responses by grounding them in retrieved documents, ensuring that the information provided is more accurate and reliable.

Query 2: "Tell me about RAG and its benefits"
Semantic Cache Hit! (similarity: 0.947)
  Matched query: "What is RAG and how does it help?"
Answer: Retrieval-Augmented Generation (RAG) is a method that combines retrieval systems with language models to enhance the generation of responses. One of the key benefits of RAG is its ability to reduce hallucinations—instances where the model generates incorrect or nonsensical information—by grounding responses in retrieved documents. This grounding helps ensure that the information provided is more accurate and relevant.

Additiona

In [20]:
# Cleanup
import os
import subprocess

os.remove("sample_docs.txt")
subprocess.run(["docker", "rm", "-f", "redis-cache"], capture_output=True)
print("Cleanup complete. Redis container stopped.")

Cleanup complete. Redis container stopped.
